In [ ]:
!nvidia-smi

Sat Jun 21 11:11:12 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

fatal: destination path 'rapidsai-csp-utils' already exists and is not an empty directory.
Installing RAPIDS remaining 25.04 libraries
Using Python 3.11.13 environment at: /usr
Audited 11 packages in 105ms

        ***********************************************************************
        The pip install of RAPIDS is complete.

        Please do not run any further installation from the conda based installation methods, as they may cause issues!

        Please ensure that you're pulling from the git repo to remain updated with the latest working install scripts.

        Troubleshooting:
            - If there is an installation failure, please check back on RAPIDSAI owned templates/notebooks to see how to update your personal files.
            - If an installation failure persists when using the latest script, please make an issue on https://github.com/rapidsai-community/rapidsai-csp-utils
        ***********************************************************************
        


In [ ]:
import cudf
%load_ext cudf.pandas

In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import time
import os

# --- Configuration ---
file_name = "large_data.csv"
num_rows = 5_000_000  # 5 million rows - adjust based on your RAM
num_unique_categories = 100
num_unique_cities = 500

# Create a large dummy CSV file if it doesn't exist

print(f"Creating a large dataframe with {num_rows} rows...")
data = {
    "id": np.arange(num_rows),
    "value": np.random.rand(num_rows) * 1000,
    "category": np.random.choice([f"cat_{i}" for i in range(num_unique_categories)], num_rows),
    "city": np.random.choice([f"city_{i}" for i in range(num_unique_cities)], num_rows),
    "timestamp": pd.to_datetime(np.random.randint(0, 365 * 24 * 60 * 60, num_rows), unit='s', origin='2020-01-01')
}
df_pd=pd.DataFrame(data)

# --- Pandas Execution (unchanged for direct comparison) ---
print("\n--- Pandas Execution ---")
start_time_pandas = time.time()

# 1. Read data
#df_pd = pd.read_csv(file_name)
print(f"Pandas initial shape: {df_pd.shape}")
print(df_pd.head())
# print(df_pd.head()) # Commented out to reduce print time if not needed

# 2. Filter data (e.g., value > 500 AND category starts with 'cat_5' AND city is 'city_10')
df_pd_filtered = df_pd[
    (df_pd["value"] > 500) &
    (df_pd["category"].str.startswith("cat_5")) &
    (df_pd["city"] == "city_10")
]

# 3. Group by and aggregate (mean of 'value' by 'category' and 'city')
df_pd_grouped = df_pd_filtered.groupby(["category", "city"])["value"].mean().reset_index()

# 4. Add a new derived column (e.g., scaled_value)
df_pd_grouped["scaled_value"] = df_pd_grouped["value"] * 1.5
print(df_pd_grouped.head())

end_time_pandas = time.time()
pandas_execution_time = end_time_pandas - start_time_pandas
print(f"Pandas total execution time with cudf.pandas extension: {pandas_execution_time:.4f} seconds")

Creating a large dataframe with 5000000 rows...

--- Pandas Execution ---
Pandas initial shape: (5000000, 5)
   id       value category      city           timestamp
0   0  437.184151   cat_69  city_170 2020-10-29 14:03:41
1   1  256.017864   cat_26  city_169 2020-05-05 17:39:18
2   2   94.997152   cat_73  city_356 2020-01-13 06:43:47
3   3  538.165807   cat_83  city_277 2020-03-21 00:15:21
4   4   93.630084   cat_26   city_72 2020-10-25 14:49:37
  category     city       value  scaled_value
0    cat_5  city_10  728.725637   1093.088456
1   cat_50  city_10  734.000701   1101.001052
2   cat_51  city_10  724.549017   1086.823525
3   cat_52  city_10  759.846522   1139.769782
4   cat_53  city_10  754.604636   1131.906954
Pandas total execution time with cudf.pandas extension: 0.1869 seconds
